In [1]:
import pandas as pd
df = pd.read_csv("data/loja_livro_filmes.csv")
df

,pais,id_cliente,data_pedido,data_chegada,tipo_compra,numero_pedido,tamanho_pacote,unidades,custo_empacotamento,custo_envio,tipo_desconto,categoria,tipo_consumo,tipo_cliente,custo_produto
0,Argentina,111962928,2002-02-15,2002-03-10,Normal,98091426,Pacote pequeno,1,3.0,5.99,Sem desconto,BlueRay,Alto Consumo,B2C,8.99
1,Argentina,145148150,2004-11-05,2004-11-26,Normal,98087803,Pacote pequeno,3,1.0,6.99,Sem desconto,BlueRay,Alto Consumo,B2C,7.99
2,Argentina,194622983,2014-09-14,2014-09-28,Normal,98072296,Pacote pequeno,1,3.0,6.99,Sem desconto,BlueRay,Alto Consumo,B2C,9.99
3,Argentina,220834258,2011-07-20,2011-07-30,Devolução,98077986,Pacote pequeno,2,1.0,3.86,Cyber Monday,BlueRay,Alto Consumo,B2B,4.86
4,Argentina,220834258,2013-09-15,2017-02-15,Normal,98074234,Pacote pequeno,1,2.0,4.88,Black Friday,BlueRay,Alto Consumo,B2B,6.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24885,Venezuela,890670789,2006-05-05,2006-05-18,Devolução,98085835,Pacote pequeno,2,1.0,6.99,Sem desconto,Livro,Alto Consumo,B2C,7.99
24886,Venezuela,892274535,2002-02-20,2002-03-08,Normal,98091414,Pacote pequeno,1,2.0,6.25,Cyber Monday,Livro,Alto Consumo,B2B,8.25
24887,Venezuela,892274535,2015-01-26,2015-02-17,Normal,98071516,Pacote pequeno,3,3.0,4.65,Sem desconto,Livro,Alto Consumo,B2B,7.65
24888,Venezuela,898039697,2002-08-22,2002-09-06,Normal,98090720,Pacote pequeno,2,2.0,4.58,Sem desconto,Livro,Alto Consumo,B2B,6.58


> O primeiro desafio é construir uma visualização que permita identificar quais países mais solicitaram produtos por pedidos para que assim possa ser feito um estudo quanto a distribuição e logística de produtos.



In [2]:
df_top10 = (df.groupby("pais")["unidades"]
    .sum()
    .nlargest(10)
    .reset_index()
)
df_top10

,pais,unidades
0,Argentina,5591
1,Equador,5566
2,Paraguai,5564
3,Peru,5550
4,Venezuela,5529
5,Colômbia,5504
6,Brasil,5473
7,Uruguai,5377
8,Chile,5304


In [3]:
df_top10.columns = ["País", "Unidades Pedidas"]
df_top10["Rank"] = df_top10.index + 1
df_top10.set_index(keys="Rank", inplace=True)

In [4]:
df_top10.style.format(
    {"Unidades Pedidas": "{} prod."}
)

,País,Unidades Pedidas
Rank,,
1,Argentina,5591 prod.
2,Equador,5566 prod.
3,Paraguai,5564 prod.
4,Peru,5550 prod.
5,Venezuela,5529 prod.
6,Colômbia,5504 prod.
7,Brasil,5473 prod.
8,Uruguai,5377 prod.
9,Chile,5304 prod.


    O objetivo desse desafio é construir uma visualização que permita identificar quais categorias de produto geraram mais e menos custos para a loja.

In [5]:
df.head(0)

# Os custos da lojo estão em custo_empacotamento, custo_envio

,pais,id_cliente,data_pedido,data_chegada,tipo_compra,numero_pedido,tamanho_pacote,unidades,custo_empacotamento,custo_envio,tipo_desconto,categoria,tipo_consumo,tipo_cliente,custo_produto


In [6]:
(df["custo_empacotamento"] + df["custo_envio"] == df["custo_produto"]).sum() == df.shape[0]

np.False_

Tá meio estranho o levantamento do custo da loja...

In [7]:
df_custo_produto = df.groupby("categoria")[["custo_produto"]].sum()
df_custo_produto.index.name = "Categoria Produto"
df_custo_produto.columns = ["Custo de Pedidos"]
df_custo_produto

,Custo de Pedidos
Categoria Produto,
BlueRay,14191.36
CD,10514.28
Coleção,73128.59
Livro,78213.80


In [8]:
# Objeto Style
table_custo_produto = df_custo_produto.style.format("R$ {:,.2f}")
table_custo_produto

,Custo de Pedidos
Categoria Produto,
BlueRay,"R$ 14,191.36"
CD,"R$ 10,514.28"
Coleção,"R$ 73,128.59"
Livro,"R$ 78,213.80"


In [9]:
table_custo_produto.highlight_max(color="red").highlight_min(color="lightgreen")

,Custo de Pedidos
Categoria Produto,
BlueRay,"R$ 14,191.36"
CD,"R$ 10,514.28"
Coleção,"R$ 73,128.59"
Livro,"R$ 78,213.80"


---

***Solução Interessante do Fórum***

```python
custo_por_categoria = df.groupby('categoria')[['custo_empacotamento','custo_envio','custo_produto']].sum()
custo_por_categoria['custo_total'] = custo_por_categoria.sum(axis=1)
custo_por_categoria = custo_por_categoria.sort_values('custo_total', ascending=False).reset_index()

estilizacao = (
    custo_por_categoria.style
        .format({
            'custo_empacotamento': 'R$ {:,.2f}',
            'custo_envio': 'R$ {:,.2f}',
            'custo_produto': 'R$ {:,.2f}',
            'custo_total': 'R$ {:,.2f}'
        })
        .background_gradient(subset=['custo_total'], cmap='Blues')
        .apply(lambda col: ['font-weight: bold' if v==col.max() else '' for v in col], subset=['custo_total'])
        .set_caption("Custos por Categoria - Acumulado")
)

display(estilizacao)
```

---